# "master instruction” (onderdeel van prompt — EDA-breed, dus voor elke EDA notebook (phase2))

Je werkt als senior research assistant voor een masterthesis in data analysis. Voor meer informatie over de thesis / onderzoeksvoorstel / opzet: bekijk bron "Geannoteerd_onderzoeksvoorstel.md" en voor extra gelinkte literatuur bron; "External factors and SHAP in Urban Parking copy.pdf" (beide bestanden zijn te vinden in de map 'literatuur_en_info' (binnen dit project)) 

voor structuur en gewenste flow check projectalomvattede: "README.md"

Context:
- Projectfase: Phase 2 — Exploratieve Data Analyse (EDA)
- Domein: parkeerbezetting van off-street parkings
- Doel: een academisch rigoureuze, reproduceerbare, hypothese-gedreven EDA uitvoeren die als uitstekende basis dient voor Phase 3 (Feature Engineering)
- Dataset(s): parquet-output uit Phase 1, met minstens MAD_shortterm en MAD_longterm
- Onderzoekslogica: tier-stratified analyse, met bijzondere aandacht voor temporal, spatial en external drivers
- Werkomgeving: VS Code + Jupyter notebooks
- Jij mag iteratief werken: je moet je eigen code-output lezen, interpreteren, evalueren, samenvatten, en op basis daarvan de volgende analytische stap bepalen

Belangrijke werkinstructies:
1. Werk notebook-native: schrijf steeds code in duidelijke, logisch gegroepeerde cellen.
2. Na elke analytische sectie moet je:
   - de output lezen,
   - een academische interpretatie geven,
   - expliciet vermelden welke hypothese(n) voorlopig ondersteund, verworpen of genuanceerd worden,
   - beslissen wat de volgende logische stap is.
3. Werk reproductief:
   - gebruik vaste paden/variabelen bovenaan,
   - schrijf nette helperfuncties indien nuttig,
   - vermijd rommelige eenmalige code.
4. Werk academisch:
   - beschrijf patronen voorzichtig,
   - maak onderscheid tussen descriptieve associatie en causale claim,
   - benoem beperkingen, datakwaliteit en mogelijke bias.
5. Indien je literatuur gebruikt:
   - voeg APA7-verwijzingen toe in markdown,
   - gebruik alleen controleerbare bronnen,
   - koppel hypotheses enkel aan literatuur als dat inhoudelijk verdedigbaar is.
6. Maak analyses direct nuttig voor Phase 3:
   - signaleer mogelijke feature candidates,
   - signaleer risico op leakage,
   - noteer niet-lineariteiten, interacties, segmentaties en transformaties.
7. Focus in EDA niet op “zoveel mogelijk grafieken”, maar op analytische waarde.
8. Rapporteer steeds ook wat NIET overtuigend blijkt.
9. Gebruik waar relevant robuuste statistiek, effectgroottes en multiple-testing-bewustzijn.
10. Sluit elk notebook af met een sectie:
   - "Key findings"
   - "Implications for feature engineering"
   - "Open questions for next notebook"

Wanneer je literatuur gebruikt om een hypothese te motiveren:
- gebruik alleen bronnen die inhoudelijk echt passen bij parkeerbezetting, mobiliteit, weersinvloeden, events, forecasting of XAI;
- label speculatieve hypothesen expliciet als speculatief maar toetsbaar;
- geef APA7-verwijzingen in markdown;
- vermijd het doen alsof literatuur causale evidentie levert wanneer het eigenlijk om associatieve studies gaat;
- als de data de literatuur niet ondersteunen, rapporteer dat eerlijk.

Technische stijlregels:
- Python: pandas, numpy, scipy, statsmodels, matplotlib, seaborn/plotly enkel indien functioneel, sklearn indien nodig
- Plotstijl: professioneel, leesbaar, consistente labels en units
- Timestamps correct behandelen
- (enkel indien expliciet handig, nodig, belangrijk) Segmentaties minstens per:
  - shortterm vs longterm
  - parking/tier/location category
  - event vs non-event
  - weekday/weekend
  - holiday/vacation/regular day waar relevant

Schrijf elke interpretatieve markdown-sectie alsof ze later kan worden herwerkt tot tekst voor de masterthesis.

Stijlregels:
- helder, academisch, voorzichtig
- geen losse bullet dump als lopende tekst beter is
- benoem richting, grootteorde, onzekerheid en beperking
- maak expliciet waarom het resultaat relevant is voor de volgende fase

Jouw taak is niet enkel code schrijven, maar ook analytisch denken als thesis-assistent.

## Notebookspecifieke prompt
Maak notebook `eda_07_feature_engineering_bridge.ipynb`.

Doel:
Alle EDA-output vertalen naar concrete, verdedigbare beslissingen voor Phase 3 (Feature Engineering).

Voer dit uit:
1. Lees de belangrijkste conclusies uit de eerdere EDA-notebooks samen.
2. Maak een gestructureerde feature recommendation table met kolommen:
   - candidate feature
   - bronkolom(men)
   - rationale vanuit EDA
   - verwacht effecttype
   - aanbevolen transformatie
   - risico op leakage
   - opnemen in forecast track?
   - opnemen in policy track?
   - opmerkingen
3. Beslis over:
   - cyclische encodings
   - lags
   - rolling features
   - event cascade features
   - weather bins / splines / thresholds
   - holiday/vacation codering
   - tier/parking encodings
   - capacity-normalisaties
   - interaction terms
4. Benoem expliciet welke features NIET aanbevolen zijn, en waarom.
5. Maak een lijst met data quality guardrails voor feature engineering.
6. Schrijf op basis van de EDA een Phase 3 hand-off memo in academische stijl.
7. Eindig met een sectie:
   - "Final EDA conclusions"
   - "Recommended feature families"
   - "Features to avoid"
   - "Open risks before modelling"

Daaronder wil ik:
8. Splitten van holdout en test + MOTIVEREN WAAROM JE WELKE KEUZE MAAKT. Blijf super kritisch, rigoureus, doelgericht op wat de thesis wilt bereiken!! en dat je de gesplitte datasets ook nog kort eens beschrijft intuitief en super helder en klaar wat de volgende stappen er mee zijn, welke regels er op vantoepassing zijn etc. leg ook uit waarom het nuttig en goed is om de split nu al te maken, en welke dataset waarvoor zal gebruikt worden

Belangrijk:
- Maak onderscheid tussen forecast track en policy track.
- Verbind alles expliciet met bevindingen uit de EDA.


## 0. Setup en scope voor Phase 3-bridge
Deze notebook vertaalt de EDA van `eda_00` t/m `eda_06` naar concrete feature engineering beslissingen.

Kernprincipes:
- Primaire modelbasis: `shortterm` (zoals onderbouwd in `eda_05` en `eda_06`).
- `longterm` blijft een secundaire robustness-context, niet de hoofdtrack.
- Forecast track en policy track krijgen dezelfde temporele split, maar verschillende featuretoegang.
- Alle transformaties moeten train-only gefit worden om leakage te vermijden.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Project root not found")


def as_flag(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0).gt(0)


def build_quality_mask(df: pd.DataFrame) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    for col in ["system_blackout", "low_data_coverage", "partial_year", "flag_occ_inconsistent"]:
        if col in df.columns:
            mask &= ~as_flag(df[col])
    mask &= pd.to_numeric(df["occupancy_rate"], errors="coerce").between(0, 1, inclusive="both")
    return mask


def extract_points_from_markdown_heading(notebook_path: Path, heading: str) -> list[str]:
    heading_norm = heading.strip().lower()
    nb = json.loads(notebook_path.read_text())

    for cell in nb["cells"]:
        if cell.get("cell_type") != "markdown":
            continue

        lines = [line.rstrip() for line in "".join(cell.get("source", "")).splitlines()]
        lines = [line for line in lines if line.strip()]
        if not lines:
            continue

        if lines[0].strip().lower() != heading_norm:
            continue

        body = lines[1:]
        points = []
        current = ""

        for line in body:
            stripped = line.strip()
            if re.match(r"^(\d+\.|-)\s+", stripped):
                if current:
                    points.append(current.strip())
                current = re.sub(r"^(\d+\.|-)\s+", "", stripped).strip()
            else:
                if current:
                    current += " " + stripped
                else:
                    points.append(stripped)

        if current:
            points.append(current.strip())

        points = [p for p in points if p]
        if points:
            return points

        plain = " ".join(body).strip()
        return [plain] if plain else []

    return []


PROJECT_ROOT = find_project_root()

# === AUTO-EXPORT ARTIFACTS (figures + displayed tables) ===
NOTEBOOK_SLUG = "eda_07_feature_engineering_bridge"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "phase2" / NOTEBOOK_SLUG
FIG_DIR = ARTIFACTS_DIR / "figures"
TABLE_DIR = ARTIFACTS_DIR / "tables"
LOG_DIR = ARTIFACTS_DIR / "logs"

for _d in [ARTIFACTS_DIR, FIG_DIR, TABLE_DIR, LOG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)


def _safe_artifact_name(name: str) -> str:
    allowed = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-")
    s = "".join(ch if ch in allowed else "_" for ch in str(name))
    while "__" in s:
        s = s.replace("__", "_")
    s = s.strip("_")
    return s or "artifact"


def save_dataframe_artifact(df: pd.DataFrame, name: str, index: bool = True) -> dict[str, str | None]:
    base = _safe_artifact_name(name)
    csv_path = TABLE_DIR / f"{base}.csv"
    parquet_path = TABLE_DIR / f"{base}.parquet"

    df.to_csv(csv_path, index=index)
    parquet_ok = True
    try:
        df.to_parquet(parquet_path, index=index)
    except Exception:
        parquet_ok = False

    return {
        "csv": str(csv_path),
        "parquet": str(parquet_path) if parquet_ok else None,
    }


if not globals().get("_DISPLAY_AUTO_EXPORT_PATCHED", False):
    _DISPLAY_AUTO_EXPORT_PATCHED = True
    _ORIG_DISPLAY = display
    _DISPLAY_COUNTER = {"n": 0}

    def display(*objs, **kwargs):
        for obj in objs:
            if isinstance(obj, pd.DataFrame):
                _DISPLAY_COUNTER["n"] += 1
                save_dataframe_artifact(obj, f"display_{_DISPLAY_COUNTER['n']:03d}", index=True)
        return _ORIG_DISPLAY(*objs, **kwargs)


try:
    import matplotlib.pyplot as plt  # noqa: F401

    if not getattr(plt, "_AUTO_EXPORT_PATCHED", False):
        _ORIG_PLT_SHOW = plt.show
        _FIG_COUNTER = {"n": 0}

        def _show_and_save(*args, **kwargs):
            fig_nums = list(plt.get_fignums())
            for fig_num in fig_nums:
                fig = plt.figure(fig_num)
                _FIG_COUNTER["n"] += 1
                fig_path = FIG_DIR / f"fig_{_FIG_COUNTER['n']:03d}.png"
                fig.savefig(fig_path, dpi=150, bbox_inches="tight")
            return _ORIG_PLT_SHOW(*args, **kwargs)

        plt.show = _show_and_save
        plt._AUTO_EXPORT_PATCHED = True
    FIG_EXPORT_ENABLED = True
except Exception:
    FIG_EXPORT_ENABLED = False

print(f"Artifacts directory: {ARTIFACTS_DIR}")
print(f"- Figures: {FIG_DIR}")
print(f"- Tables: {TABLE_DIR}")

DATA_DIR = PROJECT_ROOT / "data_processed"
NB_DIR = PROJECT_ROOT / "notebooks" / "phase2"

paths = {
    "shortterm": DATA_DIR / "MAD_shortterm.parquet",
    "longterm": DATA_DIR / "MAD_longterm.parquet",
}

season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "lente", 4: "lente", 5: "lente",
    6: "zomer", 7: "zomer", 8: "zomer",
    9: "herfst", 10: "herfst", 11: "herfst",
}

raw_dfs = {}
filtered_dfs = {}
for label, path in paths.items():
    df = pd.read_parquet(path).copy()
    df["dataset_label"] = label
    df["rounded_hour"] = pd.to_datetime(df["rounded_hour"], errors="coerce")
    df["date_only"] = pd.to_datetime(df["date_only"], errors="coerce")
    df["weekday_int"] = pd.to_numeric(df["weekday_int"], errors="coerce")
    df["is_weekend"] = df["weekday_int"].isin([5, 6])
    df["tier_temporal"] = np.where(df["parking_location_category"].astype(str).eq("centrum"), "centrum", "vesten_of_rand")
    df["season"] = df["month"].map(season_map)

    raw_dfs[label] = df
    filtered_dfs[label] = df.loc[build_quality_mask(df)].copy()

st = filtered_dfs["shortterm"].copy()
lt = filtered_dfs["longterm"].copy()

print("shortterm quality-filtered:", st.shape)
print("longterm quality-filtered:", lt.shape)
print("ST years:", sorted(st["year"].dropna().unique().tolist()))
print("LT years:", sorted(lt["year"].dropna().unique().tolist()))


Artifacts directory: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_07_feature_engineering_bridge
- Figures: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_07_feature_engineering_bridge/figures
- Tables: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/artifacts/phase2/eda_07_feature_engineering_bridge/tables
shortterm quality-filtered: (250437, 70)
longterm quality-filtered: (46643, 69)
ST years: [2019, 2020, 2023, 2024, 2025]
LT years: [2024]


## 1. Synthese van eerdere EDA-conclusies (bron voor Phase 3 beslissingen)
Hier lezen we expliciet de belangrijkste conclusies uit eerdere notebooks in, zodat featurekeuzes traceerbaar blijven naar EDA-evidentie.

In [2]:
summary_specs = [
    {
        "notebook": "eda_00_protocol_and_data_audit.ipynb",
        "sections": [
            "## Data audit conclusions",
            "## Risks for interpretation",
            "## Implications for downstream EDA and feature engineering",
        ],
    },
    {
        "notebook": "eda_01_global_descriptives.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_02_temporal_patterns.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_03_spatial_patterns.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_04_external_factors.ipynb",
        "sections": ["## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_05_shortterm_vs_longterm.ipynb",
        "sections": ["## Hypothese-overzicht", "## Key findings", "## Implications for feature engineering"],
    },
    {
        "notebook": "eda_06_interactions_and_hypothesis_synthesis.ipynb",
        "sections": ["## EDA decisions that must carry forward into Phase 3", "## Key findings", "## Implications for feature engineering"],
    },
]

prior_rows = []
for spec in summary_specs:
    nb_path = NB_DIR / spec["notebook"]
    for section in spec["sections"]:
        points = extract_points_from_markdown_heading(nb_path, section)
        if not points:
            prior_rows.append(
                {
                    "source_notebook": spec["notebook"],
                    "section": section,
                    "point_id": 1,
                    "statement": "[Geen expliciete points gevonden - manueel nakijken indien nodig]",
                }
            )
            continue

        for i, point in enumerate(points, start=1):
            prior_rows.append(
                {
                    "source_notebook": spec["notebook"],
                    "section": section,
                    "point_id": i,
                    "statement": point,
                }
            )

prior_findings_df = pd.DataFrame(prior_rows)

prior_compact_df = (
    prior_findings_df.groupby(["source_notebook", "section"], as_index=False)
    .agg(
        n_points=("statement", "count"),
        first_point=("statement", "first"),
    )
    .sort_values(["source_notebook", "section"])
)

print("Compact overzicht van EDA-conclusies")
display(prior_compact_df)

print("Volledige traceerbare conclusies")
display(prior_findings_df)


Compact overzicht van EDA-conclusies


,source_notebook,section,n_points,first_point
0,eda_00_protocol_and_data_audit.ipynb,## Data audit conclusions,1,"De data zijn bruikbaar als EDA-basis, met ster..."
1,eda_00_protocol_and_data_audit.ipynb,## Implications for downstream EDA and feature...,4,Bouw elke volgende notebook met dual reporting...
2,eda_00_protocol_and_data_audit.ipynb,## Risks for interpretation,4,Tijdsfragmentatie kan trend- en seizoensinterp...
3,eda_01_global_descriptives.ipynb,## Implications for feature engineering,5,"Gebruik `occupancy_rate` als hoofdtarget, maar..."
4,eda_01_global_descriptives.ipynb,## Key findings,4,`occupancy_rate` is descriptief de meest consi...
5,eda_02_temporal_patterns.ipynb,## Implications for feature engineering,4,Prioriteer cyclische tijdsfeatures en seizoens...
6,eda_02_temporal_patterns.ipynb,## Key findings,4,De data tonen duidelijke daily en weekly tempo...
7,eda_03_spatial_patterns.ipynb,## Implications for feature engineering,4,"Combineer coarse spatial features (`tier`, `lo..."
8,eda_03_spatial_patterns.ipynb,## Key findings,4,Centrum versus outer geeft een duidelijk nivea...
9,eda_04_external_factors.ipynb,## Implications for feature engineering,4,Gebruik niet-lineaire transformaties (bins/spl...


Volledige traceerbare conclusies


,source_notebook,section,point_id,statement
0,eda_00_protocol_and_data_audit.ipynb,## Data audit conclusions,1,"De data zijn bruikbaar als EDA-basis, met ster..."
1,eda_00_protocol_and_data_audit.ipynb,## Risks for interpretation,1,Tijdsfragmentatie kan trend- en seizoensinterp...
2,eda_00_protocol_and_data_audit.ipynb,## Risks for interpretation,2,Hoge missingness in naamvelden (holiday/event ...
3,eda_00_protocol_and_data_audit.ipynb,## Risks for interpretation,3,Parking-specifieke coverageverschillen (bv. `P...
4,eda_00_protocol_and_data_audit.ipynb,## Risks for interpretation,4,Frozen-sensor flags impliceren mogelijk meetpl...
5,eda_00_protocol_and_data_audit.ipynb,## Implications for downstream EDA and feature...,1,Bouw elke volgende notebook met dual reporting...
6,eda_00_protocol_and_data_audit.ipynb,## Implications for downstream EDA and feature...,2,Voeg kwaliteitsflags expliciet toe als feature...
7,eda_00_protocol_and_data_audit.ipynb,## Implications for downstream EDA and feature...,3,"Gebruik coverage-metadata (gaps, observatievol..."
8,eda_00_protocol_and_data_audit.ipynb,## Implications for downstream EDA and feature...,4,Vermijd leakage door kwaliteitssignalen enkel ...
9,eda_01_global_descriptives.ipynb,## Key findings,1,`occupancy_rate` is descriptief de meest consi...


De EDA-synthese wijst consistent op:
1. sterke tijds- en interactiestructuur;
2. relevante maar heterogene externe drivers;
3. duidelijke spatial heterogeniteit;
4. een methodologische noodzaak om `shortterm` als primaire Phase 3-basis te gebruiken.

De volgende secties vertalen dit naar concrete featurekeuzes.

## 2. Feature recommendation table (forecast track vs policy track)
De tabel hieronder is operationeel bruikbaar voor implementatie in Phase 3.

In [3]:
feature_rows = [
    {
        "candidate feature": "hour_sin, hour_cos",
        "bronkolom(men)": "hour",
        "rationale vanuit EDA": "Sterke intradagstructuur en 24u-seizoenscomponent in EDA 02/06.",
        "verwacht effecttype": "cyclisch, niet-lineair",
        "aanbevolen transformatie": "sin/cos encoding met periode 24",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "basisfeature, geen tuningafhankelijkheid",
    },
    {
        "candidate feature": "weekday_sin, weekday_cos",
        "bronkolom(men)": "weekday_int",
        "rationale vanuit EDA": "Wekelijkse periodiciteit (168u) is duidelijk (EDA 02).",
        "verwacht effecttype": "cyclisch",
        "aanbevolen transformatie": "sin/cos encoding met periode 7",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "combineer met weekendindicator",
    },
    {
        "candidate feature": "is_weekend",
        "bronkolom(men)": "weekday_int",
        "rationale vanuit EDA": "Dagtypeverschillen zijn zichtbaar, vooral in combinatie met events.",
        "verwacht effecttype": "niveau-shift / interactief",
        "aanbevolen transformatie": "binaire indicator",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "niet als enige dagtypefeature gebruiken",
    },
    {
        "candidate feature": "month_sin, month_cos",
        "bronkolom(men)": "month",
        "rationale vanuit EDA": "Seizoenseffecten en jaar-maandvariatie aangetoond (EDA 02/04).",
        "verwacht effecttype": "seizoensgebonden",
        "aanbevolen transformatie": "sin/cos encoding met periode 12",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "complementair met season-categorie",
    },
    {
        "candidate feature": "season",
        "bronkolom(men)": "month",
        "rationale vanuit EDA": "Weather-effecten zijn seizoensafhankelijk (EDA 04/06).",
        "verwacht effecttype": "interactie-modulator",
        "aanbevolen transformatie": "categorische encoding",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "nodig voor weather x season",
    },
    {
        "candidate feature": "occupancy_rate_lag_1h, lag_24h, lag_168h",
        "bronkolom(men)": "occupancy_rate (historisch)",
        "rationale vanuit EDA": "Hoge autocorrelatie op 24u en 168u, sterk voorspellend (EDA 02).",
        "verwacht effecttype": "autoregressief, sterk",
        "aanbevolen transformatie": "strict past-only lags per parking",
        "risico op leakage": "hoog indien fout geimplementeerd",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "nee",
        "opmerkingen": "policy track expliciet zonder occupancy-lags",
    },
    {
        "candidate feature": "rolling_mean_3h, 6h, 24h",
        "bronkolom(men)": "occupancy_rate (historisch)",
        "rationale vanuit EDA": "Stabiliteits- en regimecomponenten aanwezig (EDA 01/02).",
        "verwacht effecttype": "smoothing / trend",
        "aanbevolen transformatie": "causale rolling windows per parking",
        "risico op leakage": "hoog bij centered windows",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "nee",
        "opmerkingen": "alleen met left-closed windows",
    },
    {
        "candidate feature": "rolling_std_24h",
        "bronkolom(men)": "occupancy_rate (historisch)",
        "rationale vanuit EDA": "Volatiliteit verschilt per parking/regime (EDA 01/05).",
        "verwacht effecttype": "regime-indicator",
        "aanbevolen transformatie": "causale rolling std per parking",
        "risico op leakage": "hoog bij lookahead",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "nee",
        "opmerkingen": "handig voor piekregimes",
    },
    {
        "candidate feature": "is_event_day",
        "bronkolom(men)": "is_event_day",
        "rationale vanuit EDA": "Eventeffecten zijn segmentafhankelijk maar relevant (EDA 04/06).",
        "verwacht effecttype": "drempelmatig / interactief",
        "aanbevolen transformatie": "binaire indicator",
        "risico op leakage": "laag mits eventkalender ex ante bekend",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "event x tier en event x hour aanbevolen",
    },
    {
        "candidate feature": "event_type_flags",
        "bronkolom(men)": "is_football_day, is_sport_day, is_festival_day, ...",
        "rationale vanuit EDA": "Effecten verschillen per eventtype (EDA 04).",
        "verwacht effecttype": "heterogeen",
        "aanbevolen transformatie": "multi-hot binaire features",
        "risico op leakage": "laag-middel (afhankelijk van kalenderbeschikbaarheid)",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "sparse types eventueel clusteren",
    },
    {
        "candidate feature": "event_scale_max_binned",
        "bronkolom(men)": "event_scale_max",
        "rationale vanuit EDA": "Eventimpact schaalt niet strikt lineair met eventomvang.",
        "verwacht effecttype": "niet-lineair",
        "aanbevolen transformatie": "ordinale bins (small/medium/large)",
        "risico op leakage": "middel",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "binranden train-only bepalen",
    },
    {
        "candidate feature": "n_concurrent_events_capped",
        "bronkolom(men)": "n_concurrent_events",
        "rationale vanuit EDA": "Gelijktijdige events kunnen gecombineerde druk verhogen (EDA 04).",
        "verwacht effecttype": "monotoon of saturerend",
        "aanbevolen transformatie": "capping + mogelijk spline",
        "risico op leakage": "middel",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "kwaliteit eventfeed monitoren",
    },
    {
        "candidate feature": "football_kickoff_proximity",
        "bronkolom(men)": "football_kickoff_hour, rounded_hour",
        "rationale vanuit EDA": "Voor/na-eventprofielen zijn belangrijker dan daggemiddelden (EDA 04).",
        "verwacht effecttype": "asymmetrisch rond eventtijd",
        "aanbevolen transformatie": "lead/lag vensterfeatures in uren",
        "risico op leakage": "middel",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "alleen indien kickofftijd vooraf beschikbaar",
    },
    {
        "candidate feature": "holiday_family_flags",
        "bronkolom(men)": "is_any_holiday, is_national_holiday, is_other_holiday",
        "rationale vanuit EDA": "Holidayeffecten bestaan maar verschillen per segment (EDA 04/06).",
        "verwacht effecttype": "niveau + interactie",
        "aanbevolen transformatie": "binaire flags + eventueel hiërarchische codering",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "niet enkel aggregate holiday gebruiken",
    },
    {
        "candidate feature": "is_school_vacation",
        "bronkolom(men)": "is_school_vacation",
        "rationale vanuit EDA": "Effect aanwezig maar minder stabiel dan eventeffect.",
        "verwacht effecttype": "drempelmatig / interactief",
        "aanbevolen transformatie": "binaire indicator",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "regelmatig in sensitiviteitsanalyse",
    },
    {
        "candidate feature": "weather_temp_spline",
        "bronkolom(men)": "temp_c",
        "rationale vanuit EDA": "Temperatuureffect is seizoens- en regimeafhankelijk (EDA 04/06).",
        "verwacht effecttype": "niet-lineair",
        "aanbevolen transformatie": "restricted cubic spline of quantile bins",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "combineer met season-interactie",
    },
    {
        "candidate feature": "weather_precip_bins",
        "bronkolom(men)": "precip_mm",
        "rationale vanuit EDA": "Neerslageffect zwak lineair, mogelijk drempelmatig (EDA 04).",
        "verwacht effecttype": "drempel/saturatie",
        "aanbevolen transformatie": "droog/licht/matig/sterk bins",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "incl. precip_imputed als contextflag",
    },
    {
        "candidate feature": "weather_wind_spline",
        "bronkolom(men)": "wind_speed_ms, wind_gusts_ms",
        "rationale vanuit EDA": "Windeffect relatief robuust in meerdere segmenten (EDA 04).",
        "verwacht effecttype": "mogelijk niet-lineair",
        "aanbevolen transformatie": "spline/bins + high-wind threshold",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "gusts als extra stressor",
    },
    {
        "candidate feature": "weather_sun_humidity",
        "bronkolom(men)": "sun_duration_min, humidity_pct",
        "rationale vanuit EDA": "Sun/humidity-signalen zijn contextafhankelijk (EDA 04).",
        "verwacht effecttype": "niet-lineair, interactief",
        "aanbevolen transformatie": "z-score per maand + bins",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "met kwaliteitssignalen monitoren",
    },
    {
        "candidate feature": "tier_temporal",
        "bronkolom(men)": "parking_location_category",
        "rationale vanuit EDA": "Centrum vs randverschillen zijn consistent (EDA 03/06).",
        "verwacht effecttype": "structurele heterogeniteit",
        "aanbevolen transformatie": "binaire/categorische encoding",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "basis voor meerdere interacties",
    },
    {
        "candidate feature": "parking_id (entity encoding)",
        "bronkolom(men)": "parking_id",
        "rationale vanuit EDA": "Sterke parking-specifieke heterogeniteit (EDA 01/03).",
        "verwacht effecttype": "entity baseline",
        "aanbevolen transformatie": "one-hot of target encoding (train-only)",
        "risico op leakage": "middel bij target encoding",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "target encoding enkel met leakage-safe schema",
    },
    {
        "candidate feature": "capacity_utilization_context",
        "bronkolom(men)": "total_capacity, number_of_spaces_override",
        "rationale vanuit EDA": "Capaciteit en piekdruk zijn beleidsrelevant (EDA 01/03).",
        "verwacht effecttype": "normalisatie / saturatiecontext",
        "aanbevolen transformatie": "ratio- en drempelfeatures",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "bewaak capaciteitsschommelingen",
    },
    {
        "candidate feature": "centrum_pressure_lagged",
        "bronkolom(men)": "aggregated occupancy_rate centrum (historisch)",
        "rationale vanuit EDA": "Indicaties van spillover/saturatiegedrag (EDA 03).",
        "verwacht effecttype": "systeeminteractie",
        "aanbevolen transformatie": "lagged aggregate pressure indices",
        "risico op leakage": "middel",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "voorzichtig",
        "opmerkingen": "enkel causaal opgebouwd in tijd",
    },
    {
        "candidate feature": "event_x_tier, holiday_x_tier",
        "bronkolom(men)": "event/holiday flags + tier_temporal",
        "rationale vanuit EDA": "Interactieverschillen zijn empirisch ondersteund (EDA 06).",
        "verwacht effecttype": "interactie",
        "aanbevolen transformatie": "expliciete multiplicatieve termen",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "prioritair in beide tracks",
    },
    {
        "candidate feature": "weather_x_season",
        "bronkolom(men)": "weather vars + season",
        "rationale vanuit EDA": "Sterke seizoensmodulatie van weatherdrivers (EDA 04/06).",
        "verwacht effecttype": "interactie, niet-lineair",
        "aanbevolen transformatie": "interactietermen of season-specifieke spline",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "niet reduceren tot 1 globale weathercoef",
    },
    {
        "candidate feature": "event_x_hour",
        "bronkolom(men)": "is_event_day + hour",
        "rationale vanuit EDA": "Eventeffect varieert sterk over uren (EDA 06).",
        "verwacht effecttype": "tijdsafhankelijke interactie",
        "aanbevolen transformatie": "event * (hour_sin/hour_cos) of event-time-bins",
        "risico op leakage": "laag",
        "opnemen in forecast track?": "ja",
        "opnemen in policy track?": "ja",
        "opmerkingen": "beter dan enkel event dagflag",
    },
]

feature_reco_df = pd.DataFrame(feature_rows)

print("Feature recommendation table")
display(feature_reco_df)

track_summary = (
    feature_reco_df.groupby(["opnemen in forecast track?", "opnemen in policy track?"], as_index=False)
    .size()
    .rename(columns={"size": "n_features"})
)

print("Samenvatting opnamebeslissingen")
display(track_summary)


Feature recommendation table


,candidate feature,bronkolom(men),rationale vanuit EDA,verwacht effecttype,aanbevolen transformatie,risico op leakage,opnemen in forecast track?,opnemen in policy track?,opmerkingen
0,"hour_sin, hour_cos",hour,Sterke intradagstructuur en 24u-seizoenscompon...,"cyclisch, niet-lineair",sin/cos encoding met periode 24,laag,ja,ja,"basisfeature, geen tuningafhankelijkheid"
1,"weekday_sin, weekday_cos",weekday_int,Wekelijkse periodiciteit (168u) is duidelijk (...,cyclisch,sin/cos encoding met periode 7,laag,ja,ja,combineer met weekendindicator
2,is_weekend,weekday_int,"Dagtypeverschillen zijn zichtbaar, vooral in c...",niveau-shift / interactief,binaire indicator,laag,ja,ja,niet als enige dagtypefeature gebruiken
3,"month_sin, month_cos",month,Seizoenseffecten en jaar-maandvariatie aangeto...,seizoensgebonden,sin/cos encoding met periode 12,laag,ja,ja,complementair met season-categorie
4,season,month,Weather-effecten zijn seizoensafhankelijk (EDA...,interactie-modulator,categorische encoding,laag,ja,ja,nodig voor weather x season
5,"occupancy_rate_lag_1h, lag_24h, lag_168h",occupancy_rate (historisch),"Hoge autocorrelatie op 24u en 168u, sterk voor...","autoregressief, sterk",strict past-only lags per parking,hoog indien fout geimplementeerd,ja,nee,policy track expliciet zonder occupancy-lags
6,"rolling_mean_3h, 6h, 24h",occupancy_rate (historisch),Stabiliteits- en regimecomponenten aanwezig (E...,smoothing / trend,causale rolling windows per parking,hoog bij centered windows,ja,nee,alleen met left-closed windows
7,rolling_std_24h,occupancy_rate (historisch),Volatiliteit verschilt per parking/regime (EDA...,regime-indicator,causale rolling std per parking,hoog bij lookahead,ja,nee,handig voor piekregimes
8,is_event_day,is_event_day,Eventeffecten zijn segmentafhankelijk maar rel...,drempelmatig / interactief,binaire indicator,laag mits eventkalender ex ante bekend,ja,ja,event x tier en event x hour aanbevolen
9,event_type_flags,"is_football_day, is_sport_day, is_festival_day...",Effecten verschillen per eventtype (EDA 04).,heterogeen,multi-hot binaire features,laag-middel (afhankelijk van kalenderbeschikba...,ja,ja,sparse types eventueel clusteren


Samenvatting opnamebeslissingen


,opnemen in forecast track?,opnemen in policy track?,n_features
0,ja,ja,22
1,ja,nee,3
2,ja,voorzichtig,1


## 3. Beslissingen over featurefamilies (expliciet)
Onderstaande tabel maakt de gevraagde keuzes concreet voor implementatie.

In [4]:
decision_rows = [
    {
        "beslisdomein": "cyclische encodings",
        "beslissing": "ja, verplicht",
        "scope": "forecast + policy",
        "concrete keuze": "hour(24), weekday(7), month(12) sin/cos",
        "motivatie": "Sterke dagelijkse/wekelijkse/seizoenspatronen in EDA 02",
    },
    {
        "beslisdomein": "lags",
        "beslissing": "ja voor forecast, nee voor policy",
        "scope": "track-specifiek",
        "concrete keuze": "lag 1h/24h/168h per parking, strict past-only",
        "motivatie": "Hoge autocorrelatie en forecastwinst verwacht; policy vraagt minder inertie-afhankelijkheid",
    },
    {
        "beslisdomein": "rolling features",
        "beslissing": "ja voor forecast, nee voor policy",
        "scope": "track-specifiek",
        "concrete keuze": "rolling mean/std (3h,6h,24h) causaal",
        "motivatie": "Regime- en volatiliteitsinformatie uit EDA 01/02",
    },
    {
        "beslisdomein": "event cascade features",
        "beslissing": "ja",
        "scope": "forecast + policy",
        "concrete keuze": "event_day, eventtype, n_concurrent_events, kickoff proximity",
        "motivatie": "Eventeffecten heterogeen naar type en timing (EDA 04/06)",
    },
    {
        "beslisdomein": "weather bins/splines/thresholds",
        "beslissing": "ja",
        "scope": "forecast + policy",
        "concrete keuze": "temp/wind splines, precip bins, high-wind threshold",
        "motivatie": "Niet-lineaire en seizoensafhankelijke weatherpatronen (EDA 04/06)",
    },
    {
        "beslisdomein": "holiday/vacation codering",
        "beslissing": "ja",
        "scope": "forecast + policy",
        "concrete keuze": "is_any_holiday + typeflags + is_school_vacation",
        "motivatie": "Effecten bestaan maar segmentafhankelijk; interacties nodig",
    },
    {
        "beslisdomein": "tier/parking encodings",
        "beslissing": "ja",
        "scope": "forecast + policy",
        "concrete keuze": "tier indicator + parking entity encoding",
        "motivatie": "Sterke spatial/parking heterogeniteit (EDA 03)",
    },
    {
        "beslisdomein": "capacity-normalisaties",
        "beslissing": "ja",
        "scope": "forecast + policy",
        "concrete keuze": "saturatie- en capaciteitscontextfeatures",
        "motivatie": "Piekbelasting en capaciteit spelen rol in interpretatie en beleid",
    },
    {
        "beslisdomein": "interaction terms",
        "beslissing": "ja, prioriteit",
        "scope": "forecast + policy",
        "concrete keuze": "event x tier, holiday x tier, weather x season, event x hour",
        "motivatie": "EDA 06 toont dat main effects alleen onvoldoende zijn",
    },
]

decision_df = pd.DataFrame(decision_rows)
display(decision_df)


,beslisdomein,beslissing,scope,concrete keuze,motivatie
0,cyclische encodings,"ja, verplicht",forecast + policy,"hour(24), weekday(7), month(12) sin/cos",Sterke dagelijkse/wekelijkse/seizoenspatronen ...
1,lags,"ja voor forecast, nee voor policy",track-specifiek,"lag 1h/24h/168h per parking, strict past-only",Hoge autocorrelatie en forecastwinst verwacht;...
2,rolling features,"ja voor forecast, nee voor policy",track-specifiek,"rolling mean/std (3h,6h,24h) causaal",Regime- en volatiliteitsinformatie uit EDA 01/02
3,event cascade features,ja,forecast + policy,"event_day, eventtype, n_concurrent_events, kic...",Eventeffecten heterogeen naar type en timing (...
4,weather bins/splines/thresholds,ja,forecast + policy,"temp/wind splines, precip bins, high-wind thre...",Niet-lineaire en seizoensafhankelijke weatherp...
5,holiday/vacation codering,ja,forecast + policy,is_any_holiday + typeflags + is_school_vacation,Effecten bestaan maar segmentafhankelijk; inte...
6,tier/parking encodings,ja,forecast + policy,tier indicator + parking entity encoding,Sterke spatial/parking heterogeniteit (EDA 03)
7,capacity-normalisaties,ja,forecast + policy,saturatie- en capaciteitscontextfeatures,Piekbelasting en capaciteit spelen rol in inte...
8,interaction terms,"ja, prioriteit",forecast + policy,"event x tier, holiday x tier, weather x season...",EDA 06 toont dat main effects alleen onvoldoen...


## 4. Features die NIET aanbevolen zijn
Niet elke beschikbare kolom is modelmatig verantwoord.

In [5]:
avoid_rows = [
    {
        "feature/feature family": "number_of_occupied_spaces (contemporaan)",
        "waarom niet aanbevolen": "Is inhoudelijk dezelfde targetinformatie op hetzelfde tijdstip.",
        "risico": "direct target leakage",
        "alternatief": "gebruik alleen historische lags in forecast track",
    },
    {
        "feature/feature family": "occupancy_rate (contemporaan)",
        "waarom niet aanbevolen": "Niet beschikbaar op predictietijdstip voor dezelfde timestamp.",
        "risico": "direct leakage",
        "alternatief": "past-only lags (forecast) of geen occupancy history (policy)",
    },
    {
        "feature/feature family": "parking_id_hash",
        "waarom niet aanbevolen": "Geen extra informatie t.o.v. parking_id; moeilijker interpreteerbaar.",
        "risico": "onnodige complexiteit",
        "alternatief": "parking_id entity encoding",
    },
    {
        "feature/feature family": "parking_type",
        "waarom niet aanbevolen": "Nagenoeg constante waarde (te weinig variatie).",
        "risico": "geen informatieve waarde",
        "alternatief": "tier + parking-specifieke encodings",
    },
    {
        "feature/feature family": "event_names_combined / holiday_name tekstvelden",
        "waarom niet aanbevolen": "Hoge cardinaliteit en sparsity, beperkte robuustheid.",
        "risico": "overfitting + instabiele generalisatie",
        "alternatief": "gestandaardiseerde typeflags en schaalfeatures",
    },
    {
        "feature/feature family": "system_blackout / low_data_coverage / partial_year als predictor",
        "waarom niet aanbevolen": "Primair kwaliteitscontrole, niet stabiel operationeel gedragssignaal.",
        "risico": "artefact-learning",
        "alternatief": "gebruik als filter/guardrail en rapportering",
    },
    {
        "feature/feature family": "qc_* als directe predictive features",
        "waarom niet aanbevolen": "Representeren meetkwaliteit i.p.v. vraagdynamiek.",
        "risico": "model leert sensorkwaliteit",
        "alternatief": "gebruik voor imputatie/validatie en sensitiviteitsanalyse",
    },
    {
        "feature/feature family": "centered rolling windows",
        "waarom niet aanbevolen": "Gebruiken toekomstige observaties.",
        "risico": "tijdslekkage",
        "alternatief": "strikt causale left-aligned rolling windows",
    },
    {
        "feature/feature family": "LT als primaire trainingdataset",
        "waarom niet aanbevolen": "Slechts 1 jaar en beperkte representativiteit.",
        "risico": "zwakke externe validiteit",
        "alternatief": "ST als hoofdtrack; LT als robustness-track",
    },
]

avoid_df = pd.DataFrame(avoid_rows)
display(avoid_df)


,feature/feature family,waarom niet aanbevolen,risico,alternatief
0,number_of_occupied_spaces (contemporaan),Is inhoudelijk dezelfde targetinformatie op he...,direct target leakage,gebruik alleen historische lags in forecast track
1,occupancy_rate (contemporaan),Niet beschikbaar op predictietijdstip voor dez...,direct leakage,past-only lags (forecast) of geen occupancy hi...
2,parking_id_hash,Geen extra informatie t.o.v. parking_id; moeil...,onnodige complexiteit,parking_id entity encoding
3,parking_type,Nagenoeg constante waarde (te weinig variatie).,geen informatieve waarde,tier + parking-specifieke encodings
4,event_names_combined / holiday_name tekstvelden,"Hoge cardinaliteit en sparsity, beperkte robuu...",overfitting + instabiele generalisatie,gestandaardiseerde typeflags en schaalfeatures
5,system_blackout / low_data_coverage / partial_...,"Primair kwaliteitscontrole, niet stabiel opera...",artefact-learning,gebruik als filter/guardrail en rapportering
6,qc_* als directe predictive features,Representeren meetkwaliteit i.p.v. vraagdynamiek.,model leert sensorkwaliteit,gebruik voor imputatie/validatie en sensitivit...
7,centered rolling windows,Gebruiken toekomstige observaties.,tijdslekkage,strikt causale left-aligned rolling windows
8,LT als primaire trainingdataset,Slechts 1 jaar en beperkte representativiteit.,zwakke externe validiteit,ST als hoofdtrack; LT als robustness-track


## 5. Data quality guardrails voor feature engineering
Deze guardrails zijn verplicht voordat features in Phase 3 als "model-ready" gelden.

In [6]:
guardrail_rows = [
    {
        "guardrail": "Hard quality filter",
        "regel": "exclude system_blackout, low_data_coverage, partial_year, flag_occ_inconsistent",
        "waarom": "EDA 00 toonde verhoogd interpretatierisico bij deze flags",
        "actie bij schending": "rijen uit inferentie-/trainset verwijderen",
    },
    {
        "guardrail": "Bounded target integrity",
        "regel": "occupancy_rate moet in [0,1] liggen",
        "waarom": "Fysische plausibiliteit en modelstabiliteit",
        "actie bij schending": "rijen verwijderen of broncontrole",
    },
    {
        "guardrail": "Strict temporal causality",
        "regel": "alle lags/rolling features alleen uit historisch verleden",
        "waarom": "vermijdt tijdslekkage",
        "actie bij schending": "feature herberekenen met causale window",
    },
    {
        "guardrail": "Train-only fit van transformaties",
        "regel": "bins/splines/encoders alleen op train fitten",
        "waarom": "voorkomt informatielek naar CV-folds en holdout",
        "actie bij schending": "pipeline herfitten met correcte volgorde",
    },
    {
        "guardrail": "Split integrity",
        "regel": "geen overlap in timestamps tussen train en holdout",
        "waarom": "zuivere finale evaluatie",
        "actie bij schending": "splits opnieuw definiëren",
    },
    {
        "guardrail": "CV integrity in train",
        "regel": "rolling/blocked CV gebruikt alleen validatievensters die na het trainvenster liggen",
        "waarom": "kritische maar leakage-safe modelselectie",
        "actie bij schending": "fold-definitie corrigeren",
    },
    {
        "guardrail": "Event-data availability check",
        "regel": "enkel eventfeatures gebruiken die ex ante beschikbaar zijn",
        "waarom": "operationele realisme",
        "actie bij schending": "feature downgraden of verwijderen",
    },
    {
        "guardrail": "Weather quality context",
        "regel": "monitor imputation/suspect flags (precip_imputed, humidity_suspect, sun_duration_inconsistent)",
        "waarom": "stabiliteit van weatherfeatures",
        "actie bij schending": "sensitiviteitsanalyse en robuustheidsrapport",
    },
    {
        "guardrail": "Entity coverage minimum",
        "regel": "minimum observatievolume per parking in train en holdout",
        "waarom": "voorkomt instabiele entity-effecten",
        "actie bij schending": "pooling, regularisatie of uitsluiting",
    },
]

guardrails_df = pd.DataFrame(guardrail_rows)
display(guardrails_df)

# Huidige prevalentiecontrole op quality-flags in ST
quality_flags = ["system_blackout", "low_data_coverage", "partial_year", "flag_occ_inconsistent", "flag_frozen_sensor"]
qc_rows = []
for col in quality_flags:
    if col in raw_dfs["shortterm"].columns:
        qc_rows.append(
            {
                "flag_col": col,
                "pct_in_raw_shortterm": float(as_flag(raw_dfs["shortterm"][col]).mean() * 100),
                "pct_in_filtered_shortterm": float(as_flag(st[col]).mean() * 100),
            }
        )

quality_prevalence_df = pd.DataFrame(qc_rows)
print("Flag-prevalentiecontrole ST")
display(quality_prevalence_df.round(4))


,guardrail,regel,waarom,actie bij schending
0,Hard quality filter,"exclude system_blackout, low_data_coverage, pa...",EDA 00 toonde verhoogd interpretatierisico bij...,rijen uit inferentie-/trainset verwijderen
1,Bounded target integrity,"occupancy_rate moet in [0,1] liggen",Fysische plausibiliteit en modelstabiliteit,rijen verwijderen of broncontrole
2,Strict temporal causality,alle lags/rolling features alleen uit historis...,vermijdt tijdslekkage,feature herberekenen met causale window
3,Train-only fit van transformaties,bins/splines/encoders alleen op train fitten,voorkomt informatielek naar CV-folds en holdout,pipeline herfitten met correcte volgorde
4,Split integrity,geen overlap in timestamps tussen train en hol...,zuivere finale evaluatie,splits opnieuw definiëren
5,CV integrity in train,rolling/blocked CV gebruikt alleen validatieve...,kritische maar leakage-safe modelselectie,fold-definitie corrigeren
6,Event-data availability check,enkel eventfeatures gebruiken die ex ante besc...,operationele realisme,feature downgraden of verwijderen
7,Weather quality context,monitor imputation/suspect flags (precip_imput...,stabiliteit van weatherfeatures,sensitiviteitsanalyse en robuustheidsrapport
8,Entity coverage minimum,minimum observatievolume per parking in train ...,voorkomt instabiele entity-effecten,"pooling, regularisatie of uitsluiting"


Flag-prevalentiecontrole ST


,flag_col,pct_in_raw_shortterm,pct_in_filtered_shortterm
0,system_blackout,0.9012,0.0000
1,low_data_coverage,9.2776,0.0000
2,partial_year,2.7028,0.0000
3,flag_occ_inconsistent,0.0000,0.0000
4,flag_frozen_sensor,8.5729,9.0837


## 6. Operationele split voor Phase 3 (train + holdout)
Definitieve keuze:
- `train = (2020, 2023, 2024)`
- `holdout = 2025`
- `2019` expliciet uitgesloten omdat weatherfeatures in 2019 volledig missing zijn en daardoor niet compatibel met de external feature engineering scope.

Model- en featurekeuze gebeuren via rolling/blocked CV binnen train (zonder extra vaste sub-splits buiten de holdout).

In [7]:
# Operationele split volgens thesislogica
phase3_source = st.copy()

# Verifieer expliciet weather-compatibiliteit per jaar
WEATHER_COLS = [
    "temp_c", "precip_mm", "wind_speed_ms", "wind_gusts_ms", "humidity_pct",
    "sun_duration_min", "pressure_hpa", "shortwave_wm2", "sun_intensity_wm2",
]

weather_coverage_by_year_df = (
    phase3_source.groupby("year")[WEATHER_COLS]
    .apply(lambda d: d.isna().mean() * 100)
    .reset_index()
)

phase3_source["operational_split"] = np.select(
    [
        phase3_source["year"].isin([2020, 2023, 2024]),
        phase3_source["year"].eq(2025),
    ],
    ["train", "holdout"],
    default="excluded",
)

phase3_df = phase3_source.loc[phase3_source["operational_split"].isin(["train", "holdout"])].copy()
excluded_df = phase3_source.loc[phase3_source["operational_split"] == "excluded"].copy()

# Samenvattingen
split_overview_df = (
    phase3_df.groupby("operational_split", as_index=False)
    .agg(
        n_rows=("occupancy_rate", "size"),
        n_parkings=("parking_id", "nunique"),
        date_min=("rounded_hour", "min"),
        date_max=("rounded_hour", "max"),
        mean_occupancy=("occupancy_rate", "mean"),
    )
)
split_overview_df["pct_of_phase3_scope"] = split_overview_df["n_rows"] / len(phase3_df) * 100

split_by_year_df = (
    phase3_df.groupby(["operational_split", "year"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
    .sort_values(["operational_split", "year"])
)

split_by_tier_df = (
    phase3_df.groupby(["operational_split", "tier_temporal"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
    .sort_values(["operational_split", "tier_temporal"])
)

parking_coverage_df = (
    phase3_df.groupby(["operational_split", "parking_id"], as_index=False)
    .size()
    .rename(columns={"size": "n_rows"})
)
parking_coverage_summary_df = (
    parking_coverage_df.groupby("operational_split", as_index=False)
    .agg(
        min_rows_per_parking=("n_rows", "min"),
        median_rows_per_parking=("n_rows", "median"),
        max_rows_per_parking=("n_rows", "max"),
    )
)

excluded_summary_df = (
    excluded_df.groupby("year", as_index=False)
    .agg(
        n_rows=("occupancy_rate", "size"),
        n_parkings=("parking_id", "nunique"),
        date_min=("rounded_hour", "min"),
        date_max=("rounded_hour", "max"),
    )
)
excluded_summary_df["why_excluded"] = "weather-features 100% missing -> niet compatibel met external FE"

# Binnen-train evaluatie: rolling/blocked CV (zonder extra vaste sub-splits buiten holdout)
train_df = phase3_df.loc[phase3_df["operational_split"] == "train"].copy()

fold_specs = [
    ("cv_fold_1", pd.Timestamp("2023-10-01 00:00:00"), pd.Timestamp("2023-12-31 23:00:00")),
    ("cv_fold_2", pd.Timestamp("2024-04-01 00:00:00"), pd.Timestamp("2024-06-30 23:00:00")),
    ("cv_fold_3", pd.Timestamp("2024-10-01 00:00:00"), pd.Timestamp("2024-12-31 23:00:00")),
]

cv_rows = []
for fold_id, valid_start, valid_end in fold_specs:
    valid_mask = (train_df["rounded_hour"] >= valid_start) & (train_df["rounded_hour"] <= valid_end)
    train_mask = train_df["rounded_hour"] < valid_start

    cv_rows.append(
        {
            "fold": fold_id,
            "train_end_before": valid_start,
            "valid_start": valid_start,
            "valid_end": valid_end,
            "n_train": int(train_mask.sum()),
            "n_valid": int(valid_mask.sum()),
            "n_valid_parkings": int(train_df.loc[valid_mask, "parking_id"].nunique()),
        }
    )

cv_plan_df = pd.DataFrame(cv_rows)

split_manifest_rows = [
    {
        "split": "train",
        "doel": "feature fit + model fit",
        "regels": "alle transformaties train-only; modelkeuze via rolling CV in train",
        "volgende stap": "feature pipeline bouwen en tune via cv_plan_df",
    },
    {
        "split": "holdout",
        "doel": "finale, ongeziene thesis-evaluatie op 2025",
        "regels": "geen tuning op holdoutresultaten",
        "volgende stap": "finale rapportering in Phase 4",
    },
]
split_manifest_df = pd.DataFrame(split_manifest_rows)

print("Weather missingness per jaar (in ST quality-filtered)")
display(weather_coverage_by_year_df.round(2))

print("Operationele split-overzicht")
display(split_overview_df.sort_values("operational_split").round(5))

print("Split x jaar")
display(split_by_year_df)

print("Split x tier")
display(split_by_tier_df)

print("Parking coverage per split")
display(parking_coverage_summary_df.round(2))

print("Uitgesloten data")
display(excluded_summary_df)

print("Binnen-train rolling CV plan")
display(cv_plan_df)

print("Split gebruiksmanifest")
display(split_manifest_df)


# Exporteer operationele splitbestanden voor Phase 3
SPLIT_EXPORT_DIR = PROJECT_ROOT / "data_processed" / "phase3_splits"
SPLIT_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

train_export_df = phase3_df.loc[phase3_df["operational_split"] == "train"].copy()
holdout_export_df = phase3_df.loc[phase3_df["operational_split"] == "holdout"].copy()
excluded_export_df = excluded_df.copy()

train_path = SPLIT_EXPORT_DIR / "MAD_shortterm_train_2020_2023_2024.parquet"
holdout_path = SPLIT_EXPORT_DIR / "MAD_shortterm_holdout_2025.parquet"
excluded_path = SPLIT_EXPORT_DIR / "MAD_shortterm_excluded_2019.parquet"

train_export_df.to_parquet(train_path, index=False)
holdout_export_df.to_parquet(holdout_path, index=False)
excluded_export_df.to_parquet(excluded_path, index=False)

cv_plan_path = SPLIT_EXPORT_DIR / "shortterm_train_rolling_cv_plan.csv"
cv_plan_df.to_csv(cv_plan_path, index=False)

split_summary_path = SPLIT_EXPORT_DIR / "shortterm_operational_split_summary.csv"
split_overview_df.to_csv(split_summary_path, index=False)

split_export_manifest = pd.DataFrame(
    [
        {"artifact": "train_parquet", "path": str(train_path), "n_rows": int(len(train_export_df))},
        {"artifact": "holdout_parquet", "path": str(holdout_path), "n_rows": int(len(holdout_export_df))},
        {"artifact": "excluded_2019_parquet", "path": str(excluded_path), "n_rows": int(len(excluded_export_df))},
        {"artifact": "cv_plan_csv", "path": str(cv_plan_path), "n_rows": int(len(cv_plan_df))},
        {"artifact": "split_summary_csv", "path": str(split_summary_path), "n_rows": int(len(split_overview_df))},
    ]
)

split_manifest_export_path = SPLIT_EXPORT_DIR / "split_export_manifest.csv"
split_export_manifest.to_csv(split_manifest_export_path, index=False)

print("Split exports geschreven naar:", SPLIT_EXPORT_DIR)
display(split_export_manifest)

print("Opmerking LT:")
print("Longterm blijft buiten primaire split-architectuur en dient enkel als secundaire robustness-context.")


Weather missingness per jaar (in ST quality-filtered)


,year,temp_c,precip_mm,wind_speed_ms,wind_gusts_ms,humidity_pct,sun_duration_min,pressure_hpa,shortwave_wm2,sun_intensity_wm2
0,2019,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Operationele split-overzicht


/var/folders/qb/kw39kgxn4qq33plfhj2fzq5m0000gn/T/ipykernel_93297/3492958666.py:128: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(split_overview_df.sort_values("operational_split").round(5))


,operational_split,n_rows,n_parkings,date_min,date_max,mean_occupancy,pct_of_phase3_scope
0,holdout,87600,10,2025-01-01,2025-12-31 23:00:00,0.40081,40.28753
1,train,129837,10,2020-01-01,2024-12-31 23:00:00,0.36163,59.71247


...

Split gebruiksmanifest


,split,doel,regels,volgende stap
0,train,feature fit + model fit,alle transformaties train-only; modelkeuze via...,feature pipeline bouwen en tune via cv_plan_df
1,holdout,"finale, ongeziene thesis-evaluatie op 2025",geen tuning op holdoutresultaten,finale rapportering in Phase 4


Split exports geschreven naar: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/phase3_splits


,artifact,path,n_rows
0,train_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,129837
1,holdout_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,87600
2,excluded_2019_parquet,/Users/emilevandevoorde/Documents/mp_mechelen_...,33000
3,cv_plan_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,3
4,split_summary_csv,/Users/emilevandevoorde/Documents/mp_mechelen_...,2


Opmerking LT:
Longterm blijft buiten primaire split-architectuur en dient enkel als secundaire robustness-context.


### Waarom deze aangepaste split methodologisch sterker is
1. `2025` blijft een volledig ongeziene holdout voor finale thesis-evaluatie.
2. `2019` wordt expliciet uitgesloten op basis van data-compatibiliteit: weatherfeatures zijn daar volledig missing.
3. Een aparte extra vaste sub-split buiten train is niet noodzakelijk als we binnen train een strikte rolling/blocked CV gebruiken.
4. Deze aanpak maximaliseert trainvolume voor feature learning, terwijl evaluatie toch kritisch en leakage-safe blijft.

### Forecast track versus policy track binnen dezelfde operationele split
- **Forecast track**: mag autoregressieve features gebruiken (lags/rollings), maar strikt causaal opgebouwd.
- **Policy track**: geen occupancy-lags, voor betere scenario-interpretatie.
- **Belangrijk**: beide tracks gebruiken dezelfde operationele `train/holdout`-rijen; alleen de featuretoegang verschilt.


## 7. Phase 3 hand-off memo (academische stijl)
De EDA-fase toont dat parkeerdynamiek in Mechelen sterk interactief is: temporele ritmes, spatial context en externe drivers beïnvloeden bezetting in combinatie, niet als losse lineaire effecten. Een interaction-first feature engineering ontwerp blijft daarom de meest verdedigbare keuze.

Voor de hoofdtrack wordt `shortterm` gebruikt, terwijl `longterm` enkel als secundaire robustness-context blijft bestaan. Die keuze is methodologisch coherent met coverage en representativiteit uit eerdere EDA-notebooks. Binnen `shortterm` worden cyclische tijdsfeatures, entity-context, event- en kalenderstructuren en niet-lineaire weatherfeatures centraal gezet. Autoregressieve families blijven relevant voor forecastperformantie, maar worden bewust uit de policy track gehouden.

De operationele split wordt gefixeerd als `train=(2020,2023,2024)` en `holdout=2025`. Jaar `2019` wordt uitgesloten omdat weatherfeatures daar volledig missing zijn en dus niet compatibel met de externe feature engineering scope. In plaats van extra vaste sub-splits wordt model- en featurekeuze binnen train georganiseerd via rolling/blocked tijds-CV. Hierdoor blijft de evaluatie kritisch, maar wordt onnodig verlies van trainingsdata vermeden.

Concreet voor Phase 3: (i) implementeer de aanbevolen featurefamilies per track, (ii) fit alle transformaties uitsluitend op train, (iii) evalueer featurekeuzes via tijds-CV binnen train, en (iv) reserveer 2025 strikt voor finale, ongeziene evaluatie.

## Final EDA conclusions
1. De data ondersteunen een interaction-first modellering, niet enkel main effects.
2. `shortterm` is methodologisch de juiste primaire basis voor Phase 3.
3. Forecast en policy doelen vereisen gedeelde basisfeatures maar verschillende toegang tot autoregressieve informatie.
4. De operationele split `train=(2020,2023,2024)` en `holdout=2025` is het meest verdedigbaar gezien datacompatibiliteit en evaluatiediscipline.


## Recommended feature families
1. Cyclische tijdsfeatures (`hour`, `weekday`, `month`) en season-context.
2. Event- en kalenderfeatures inclusief type-, schaal- en timingcomponenten.
3. Niet-lineaire weatherfeatures met season-interacties.
4. Spatial/entity-features (`tier`, `parking_id`) en hun interacties.
5. Forecast-only autoregressieve families (lags + causale rollings).

## Features to avoid
1. Contemporaine target-proxies (`occupancy_rate`, `number_of_occupied_spaces`) als input.
2. Pure kwaliteitsflags (`system_blackout`, `low_data_coverage`, `partial_year`) als gedragsfeatures.
3. Hoge-cardinaliteit tekstvelden (`event_names_combined`, holiday-name strings) zonder robuuste compressie.
4. Niet-causale featureconstructies (centered rolling windows, future-informed encodings).

## Open risks before modelling
1. Eventdata-kwaliteit en ex-ante beschikbaarheid kunnen featurestabiliteit beperken.
2. Structurele drift tussen 2024 en 2025 kan prestaties in holdout onder druk zetten.
3. Parking-specifieke coverage-onbalans kan entity-encodings vertekenen zonder regularisatie.
4. Overmatige interactiecomplexiteit kan overfitting veroorzaken zonder strikte validatieregels.